# CS-781 Phase 2 (L-NER) — Architecture 2: encoder-decoder v2 (143M), HF ByteLevel BPE (Phase-1 notebook archit-2-v2)

**What this notebook does**
1. Loads my Phase-1 checkpoint and my from-scratch BPE tokenizer.
2. Re-creates the exact Phase-1 text preprocessing (**`" ".join(text.split())`** + whitespace collapse) **while keeping a map back to the original characters**, so every BPE token knows its exact original `[start, end)`. It also checks that the token ids are identical to the ones Phase 1 produced.
3. Checks that gold spans → BIO tags → spans comes back exactly (the *alignment upper bound*), and that the tagger's encoder reproduces the Phase-1 model's computation exactly.
4. Runs one experiment per adaptation method. Each trains on fold1, picks the best epoch by **strict macro-F1 on fold2**, retrains on fold1+fold2, predicts fold3 and writes `submission_<name>.zip`.
5. Writes `results.json` per experiment: val F1, per-label P/R/F1, error types, params, trainable params, peak GPU memory, train/inference time.


In [1]:
# ============================================================ 1. SETUP
import os, sys, json, time, shutil, subprocess, threading, pickle
from pathlib import Path

QUICK_TEST = False        # True: tiny end-to-end check (~10 min).  False: the real run (commit it).
ARCH, NORM = "encdec2", "ws"
PHASE1_HINT = "archit-2-v2"   # substring of the attached Phase-1 notebook's input path

WORK = Path("/kaggle/working/p2"); WORK.mkdir(parents=True, exist_ok=True)
OUT  = WORK / ("runs_quick" if QUICK_TEST else "runs")

def find(name, must_contain="", anywhere=False):
    hits = sorted(p for p in Path("/kaggle/input").rglob(name) if (anywhere or PHASE1_HINT in str(p)) and must_contain in str(p))
    return str(hits[0]) if hits else None

PHASE1_CKPT = find("best_model.pt", "arch2_v2_training") or find("best_model.pt")   # plain state_dict
MODEL_PY    = "builtin"                                # the model classes are inside p2_ner.py
v = [p for p in sorted(Path("/kaggle/input").rglob("vocab.json")) if "arch2_final_processed/tokenizer" in str(p)]
v = v or [p for p in sorted(Path("/kaggle/input").rglob("vocab.json")) if (p.parent / "merges.txt").exists() and "archit-2-0" in str(p)]
TOK_PATH = str(v[0].parent) if v else None
TOK_PY = TOK_PATH
print("checkpoint :", PHASE1_CKPT); print("model code :", MODEL_PY); print("tokenizer  :", TOK_PY, "|", TOK_PATH)
assert PHASE1_CKPT and MODEL_PY, f"attach the Phase-1 notebook whose input path contains '{PHASE1_HINT}' (Add Input -> Your Work -> Notebooks), or fix PHASE1_HINT"
assert TOK_PY and TOK_PATH, "tokenizer files not found - attach the tokenizer dataset (a1_tokenization.py + vocab.json + merges.txt)"

from kaggle_secrets import UserSecretsClient
HF_TOKEN = None
for k in ["HF", "HF_TOKEN"]:
    try: HF_TOKEN = UserSecretsClient().get_secret(k); break
    except Exception: pass
assert HF_TOKEN, "add your Hugging Face token as a Kaggle secret named HF (Add-ons -> Secrets) and tick it for this notebook"

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                              allow_patterns=["Phase_2_LNER/*"], local_dir="/kaggle/working/hf_data")) / "Phase_2_LNER"
print(sorted(str(p.relative_to(DATA)) for p in DATA.rglob("*.json")))
LABELS = [l.strip() for l in open(DATA / "Train" / "labels.txt") if l.strip()]
print("labels:", LABELS)


checkpoint : /kaggle/input/notebooks/dummy595/archit-2-v2/arch2_v2_training/best_model.pt
model code : builtin
tokenizer  : /kaggle/input/notebooks/dummy595/archit-2-0/arch2_final_processed/tokenizer | /kaggle/input/notebooks/dummy595/archit-2-0/arch2_final_processed/tokenizer


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

['Test/fold3-text.json', 'Train/fold1-gold.json', 'Train/fold1-text.json', 'Train/fold2-gold.json', 'Train/fold2-text.json']
labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [2]:
%%writefile /kaggle/working/p2/p2_ner.py
"""CS-781 Phase 2 (L-NER) on top of my Phase-1 models: Transformer-XL (arch=txl), decoder-only LegalLM (arch=dec),
the encoder of the encoder-decoder (arch=encdec), or RetNet (arch=ret).

One file, used two ways:
  * imported by the notebook  -> data prep, alignment checks, model-equivalence test
  * run as a script            -> `python p2_ner.py job.json` trains/evaluates/predicts ONE experiment

Pipeline
  text --(ASCII-fold + whitespace-collapse, exactly like Phase-1 pretraining, but keeping a char map)-->
  BPE tokens with ORIGINAL character offsets --> BIO tags (25) --> 512-token sliding windows -->
  Transformer-XL encoder (+ PEFT) + token head --> overlap-weighted log-prob merge per document -->
  BIO-constrained Viterbi --> character spans (whitespace-stripped) --> prediction.json

Adaptation methods (METHOD):
  full     full fine-tuning
  partial  top-N blocks + head (layer-wise / partial fine-tuning)
  lora     LoRA:  y = W x + s*B(A x)
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025, multiplicative):  y = (I + s*B A) W x   (B = 0 -> identity at init)
  adapter  Houlsby-style bottleneck adapters after attention and FFN sub-layers
  bitfit   biases (+ norms; TXL u/v biases). Bias-free LegalLM: zero-init biases added to every projection
  kd       knowledge distillation: a fine-tuned 14-layer teacher -> a shallower student built from Phase-1 layers
"""
import json
import math
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# ============================================================== text normalisation with char map
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")
# Phase-1 pre-tokenizer WITH the underscore fix that the Phase-1 notebook applied
PRE_RE = re.compile(r" ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+")


def normalize_with_map(text):
    """ASCII mode (Transformer-XL runs): same output as the TXL Phase-1 `normalize_text` (ASCII fold,
    straight quotes, single spaces, no newlines), plus omap[i] = ORIGINAL index of normalised char i."""
    norm, st, _ = normalize_ascii_map(text)
    return norm, st


def normalize_ascii_map(text):
    out, ost, oen = [], [], []
    pend = -1                                   # orig index of the first whitespace in a pending run
    for i, ch in enumerate(text):
        s = ch
        if not ch.isascii():
            s = ch.translate(_MAP)
            if not s.isascii():
                s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
        if s and _CTRL.search(s):
            s = _CTRL.sub("", s)
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


def normalize_nfkc_map(text, delete_table):
    """NFKC mode (decoder-only run): same output as a1_tokenization.normalize_text
    (delete Cc/Cf/Cs -> NFKC -> collapse whitespace -> strip). NFKC is applied per *cluster*
    (a char + its combining marks) so compositions match the whole-string NFKC.
    ost/oen[i] = ORIGINAL [start, end) of the cluster that produced normalised char i."""
    out, ost, oen = [], [], []
    pend = -1
    n, i = len(text), 0
    while i < n:
        j = i + 1
        while j < n and unicodedata.combining(text[j]):
            j += 1
        s = unicodedata.normalize("NFKC", text[i:j].translate(delete_table))
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(j)
        i = j
    return "".join(out), ost, oen


def align_map(orig, norm):
    """Character map norm -> orig from a diff (used only when the fast cluster map disagrees)."""
    import difflib
    st, en = [0] * len(norm), [0] * len(norm)
    sm = difflib.SequenceMatcher(None, orig, norm, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        for j in range(j1, j2):
            if tag == "equal":
                st[j], en[j] = i1 + (j - j1), i1 + (j - j1) + 1
            else:                                  # replaced / inserted: map to the whole changed region
                st[j], en[j] = i1, max(i2, i1 + 1) if i2 > i1 else min(i1 + 1, len(orig))
    return st, en


# RetNet Phase-1 preprocessing: ret_data.clean_text = remove "\r", collapse 3+ newlines to 2, strip (unicode kept)
_NL3 = re.compile(r"\n{3,}")
# the ORIGINAL a1_tokenizer pre-tokenizer (no underscore fix: "_" is never matched, i.e. dropped) - used by the RetNet run
PRE_RE_ORIG = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+")


def clean_text_ret(t):
    return _NL3.sub("\n\n", t.replace("\r", "")).strip()


def normalize_clean_map(text):
    idx = [i for i, c in enumerate(text) if c != "\r"]
    keep, run = [], 0
    for i in idx:                                              # 3+ consecutive newlines -> keep the first two
        if text[i] == "\n":
            run += 1
            if run > 2:
                continue
        else:
            run = 0
        keep.append(i)
    a, b = 0, len(keep)
    while a < b and text[keep[a]].isspace():
        a += 1
    while b > a and text[keep[b - 1]].isspace():
        b -= 1
    keep = keep[a:b]
    return "".join(text[i] for i in keep), keep, [i + 1 for i in keep]


# Archit-2-v2 preprocessing: " ".join(text.split())  (whitespace runs -> one space, ends stripped, unicode kept)
def normalize_ws_map(text):
    out, ost, oen = [], [], []
    pend = -1
    for i, ch in enumerate(text):
        if ch.isspace():
            if pend < 0:
                pend = i
        else:
            if pend >= 0 and out:
                out.append(" "); ost.append(pend); oen.append(pend + 1)
            pend = -1
            out.append(ch); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


class HFBPE:
    """Archit-2-v2 tokenizer: HuggingFace `tokenizers` ByteLevelBPETokenizer(vocab.json, merges.txt)
    (<pad>0 <s>1 </s>2 <unk>3), text = " ".join(text.split()). Offsets come from the tokenizer itself."""

    def __init__(self, tok_path, norm="ws"):
        from tokenizers import ByteLevelBPETokenizer
        p = Path(tok_path)
        p = p.parent if p.is_file() else p
        self.tok = ByteLevelBPETokenizer(str(p / "vocab.json"), str(p / "merges.txt"))
        self.kind, self.norm = "hf", norm
        self.vocab_size = self.tok.get_vocab_size()
        assert [self.tok.token_to_id(t) for t in ("<pad>", "<s>", "</s>", "<unk>")] == [0, 1, 2, 3]

    def normalize(self, text):
        n, st, en = normalize_ws_map(text)
        assert n == " ".join(text.split())
        return n, st, en

    def reference_encode(self, raw_text):
        return self.tok.encode(" ".join(raw_text.split())).ids


# ============================================================== tokenizer with offsets
class BPE:
    """Uniform wrapper around my two from-scratch Phase-1 tokenizers.
      kind='old' : a1_tokenizer.py  + bpe_tokenizer.json   (<pad>0 <unk>1 <s>2 </s>3; underscore-fixed regex)
      kind='new' : a1_tokenization.py + vocab.json/merges.txt (<pad>0 <s>1 </s>2 <unk>3; GPT-2 style regex)
    norm='ascii' : TXL pretraining preprocessing (ASCII fold)   |   norm='nfkc' : a1_tokenization.normalize_text
    norm='clean' : RetNet preprocessing (\r removal, newline collapse; ORIGINAL old regex, underscores dropped)
    Exposes: pre_re, encode_piece(piece) -> ids, nbytes[id], vocab_size, normalize(text), reference_encode(text)."""

    def __init__(self, tokenizer_py, tok_path, norm="ascii"):
        import importlib.util
        name = Path(tokenizer_py).stem
        spec = importlib.util.spec_from_file_location(name + "_p2", tokenizer_py)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        self.mod, self.norm = mod, norm
        if hasattr(mod, "BPETokenizer"):                              # old tokenizer
            self.kind = "old"
            p = Path(tok_path)
            self.tok = mod.BPETokenizer.load(str(p if p.is_file() else p / "bpe_tokenizer.json"))
            self.pre_re = PRE_RE_ORIG if norm == "clean" else PRE_RE      # RetNet run used the unpatched regex
            self.encode_piece = self.tok._encode_pretoken
            self.nbytes = [len(b) for b in self.tok.id_to_bytes]
            self.vocab_size = self.tok.get_vocab_size()
        else:                                                         # new tokenizer
            self.kind = "new"
            p = Path(tok_path)
            self.tok = mod.Tokenizer.load(str(p.parent if p.is_file() else p))
            self.pre_re = mod.PRETOKEN_RE
            self.encode_piece = self.tok._encode_word
            k = mod.N_SPECIAL
            self.nbytes = [0] * k + [len(t) for t in self.tok.id_to_token[k:]]   # 1 char = 1 byte (GPT-2 byte map)
            self.vocab_size = self.tok.vocab_size
        assert (norm == "ascii") or (norm == "nfkc" and self.kind == "new") or (norm == "clean" and self.kind == "old")

    def normalize(self, text):
        if self.norm == "nfkc":
            n, st, en = normalize_nfkc_map(text, self.mod._DELETE_TABLE)
            ref = self.mod.normalize_text(text)
            if n != ref:                       # rare: NFKC interaction across clusters -> exact diff alignment
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                st, en = align_map(text, ref)
                n = ref
            return n, st, en
        if self.norm == "clean":
            n, st, en = normalize_clean_map(text)
            if n != clean_text_ret(text):
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                n = clean_text_ret(text)
                st, en = align_map(text, n)
            return n, st, en
        return normalize_ascii_map(text)

    def encode_full(self, norm_text):
        out = []
        for piece in self.pre_re.findall(norm_text):
            out.extend(self.encode_piece(piece))
        return out

    def reference_encode(self, raw_text):
        """Exactly what Phase-1 pretraining fed the model for this raw text (used only for checks)."""
        if self.norm == "nfkc":
            return self.tok.encode(raw_text)                         # normalize_text + encode
        if self.norm == "clean":                                     # ret_data.clean_text + original a1_tokenizer.encode
            t = clean_text_ret(raw_text)
            return [i for piece in PRE_RE_ORIG.findall(t) for i in self.tok._encode_pretoken(piece)]
        return self.encode_full(self.normalize(raw_text)[0])


def load_bpe(tokenizer_py, tok_path, norm="ascii"):
    if norm == "ws":
        return HFBPE(tok_path, norm)
    return BPE(tokenizer_py, tok_path, norm)


def encode_doc(tok, text):
    """-> dict(ids int32[n], cs int32[n], ce int32[n]).  For token t, the ORIGINAL-text span is [cs, ce),
    where cs skips the single leading space a BPE token may carry. Handles tokens that split a
    multi-byte UTF-8 character (byte-level BPE) by mapping byte offsets to characters."""
    norm, ost, oen = tok.normalize(text)
    ids, cs, ce = [], [], []
    if getattr(tok, "kind", "") == "hf":                         # offsets straight from the HF tokenizer
        if not norm:
            return dict(ids=np.zeros(0, np.int32), cs=np.zeros(0, np.int32), ce=np.zeros(0, np.int32))
        enc = tok.tok.encode(norm)
        for tid, (a, z) in zip(enc.ids, enc.offsets):
            z = max(z, a + 1) - 1                                # last normalised char touched by this token
            a = min(a, len(norm) - 1); z = min(max(z, a), len(norm) - 1)
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid); cs.append(ost[a2]); ce.append(oen[z])
        return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))
    for m in tok.pre_re.finditer(norm):
        piece, p0 = m.group(), m.start()
        if piece.isascii():
            b2c = None
            nb = len(piece)
        else:
            b2c = [ci for ci, ch in enumerate(piece) for _ in range(len(ch.encode("utf-8")))]
            nb = len(b2c)
        bp = 0
        for tid in tok.encode_piece(piece):
            L = tok.nbytes[tid]
            a = p0 + (bp if b2c is None else b2c[bp])
            z = p0 + ((bp + L - 1) if b2c is None else b2c[bp + L - 1])     # last char touched by this token
            bp += L
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid)
            cs.append(ost[a2])
            ce.append(oen[z])
        assert bp == nb, "token byte lengths do not add up"
    return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))


# ============================================================== labels / BIO
def make_tags(labels):
    tags = ["O"] + [f"B-{l}" for l in labels] + [f"I-{l}" for l in labels]
    return tags, {t: i for i, t in enumerate(tags)}


def strip_span(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def bio_from_gold(text, enc, ents, tag2id):
    """Token BIO ids from character-level gold entities (entities are whitespace-stripped first)."""
    owner = np.full(len(text) + 1, -1, np.int32)
    lab_of = []
    for k, e in enumerate(ents):
        s, en = strip_span(text, int(e[0]), int(e[1]))
        owner[s:en] = k
        lab_of.append(e[2])
    cs = enc["cs"]
    y = np.zeros(len(cs), np.int64)
    prev = -1
    for t in range(len(cs)):
        k = owner[cs[t]] if cs[t] < len(text) else -1
        if k >= 0 and text[cs[t]].isspace() and k != prev:     # a whitespace-only token continues an entity, never starts one
            k = -1
        if k >= 0:
            y[t] = tag2id[("I-" if k == prev else "B-") + lab_of[k]]
        prev = k
    return y


def spans_from_tags(text, enc, tag_ids, tags):
    out, cur = [], None                     # cur = [first_tok, last_tok, label]
    for t, ti in enumerate(tag_ids):
        tg = tags[ti]
        if tg == "O":
            if cur: out.append(cur); cur = None
            continue
        bi, lab = tg.split("-", 1)
        if bi == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [t, t, lab]
        else:
            cur[1] = t
    if cur: out.append(cur)
    res = []
    for a, b, lab in out:
        s, e = strip_span(text, int(enc["cs"][a]), int(enc["ce"][b]))
        if e > s:
            res.append([s, e, lab])
    return res


def bio_transitions(tags):
    """allowed[i, j] for tag i -> tag j; start_ok[j]."""
    T = len(tags)
    allowed = np.ones((T, T), bool)
    start_ok = np.ones(T, bool)
    for j, tj in enumerate(tags):
        if tj.startswith("I-"):
            lab = tj[2:]
            start_ok[j] = False
            for i, ti in enumerate(tags):
                allowed[i, j] = ti in (f"B-{lab}", f"I-{lab}")
    return allowed, start_ok


def viterbi(logp, allowed, start_ok):
    """logp [n, T] log-probs -> best BIO-valid tag sequence."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, np.int64)
    trans = np.where(allowed, 0.0, -1e9)
    score = np.where(start_ok, logp[0], -1e9)
    back = np.zeros((n, T), np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans            # [from, to]
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    best = np.zeros(n, np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best


# ============================================================== metric (strict span + label)
def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = {(int(s), int(e), l) for s, e, l, *_ in gold[did]}
        p = {(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l])
    mac = lambda k: sum(per[l][k] for l in labels) / len(labels)
    return dict(macro_f1=mac("f1"), macro_p=mac("p"), macro_r=mac("r"), per_label=per)


def error_breakdown(gold, pred):
    """Classify every false positive: label error (same span, other label), boundary error
    (overlaps a gold span of the same label), or spurious; plus the label-confusion counts."""
    kinds, conf = Counter(), Counter()
    for did in gold:
        g = [(int(s), int(e), l) for s, e, l, *_ in gold[did]]
        gs = set(g)
        for s, e, l in [(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])]:
            if (s, e, l) in gs:
                continue
            same = [gl for (a, b, gl) in g if a == s and b == e]
            if same:
                kinds["label_error"] += 1; conf[f"{same[0]}->{l}"] += 1
            elif any(a < e and s < b and gl == l for a, b, gl in g):
                kinds["boundary_error"] += 1
            elif any(a < e and s < b for a, b, gl in g):
                kinds["overlap_other_label"] += 1
            else:
                kinds["spurious"] += 1
    return dict(fp_kinds=dict(kinds), top_confusions=conf.most_common(15))


# ============================================================== data prep (called from the notebook)
def prepare(data_dir, tok, labels, cache_path):
    data_dir = Path(data_dir)
    tags, tag2id = make_tags(labels)
    out = {}
    for fold in ["fold1", "fold2", "fold3"]:
        sub = "Test" if fold == "fold3" else "Train"
        texts = json.load(open(data_dir / sub / f"{fold}-text.json", encoding="utf-8"))
        gold = json.load(open(data_dir / sub / f"{fold}-gold.json", encoding="utf-8")) if fold != "fold3" else None
        docs = {}
        for did, text in texts.items():
            enc = encode_doc(tok, text)
            d = dict(text=text, **enc)
            if gold is not None:
                d["gold"] = gold.get(did, [])
                d["y"] = bio_from_gold(text, enc, d["gold"], tag2id)
            docs[did] = d
        out[fold] = docs
    pickle.dump(dict(labels=labels, tags=tags, folds=out), open(cache_path, "wb"))
    return out


def alignment_report(folds, labels):
    """Upper bound of strict macro-F1 if the model predicted the gold BIO tags perfectly."""
    tags, _ = make_tags(labels)
    rep = {}
    for fold in ["fold1", "fold2"]:
        gold = {d: v["gold"] for d, v in folds[fold].items()}
        pred = {d: spans_from_tags(v["text"], v, v["y"], tags) for d, v in folds[fold].items()}
        sc = strict_scores(gold, pred, labels)
        miss = []
        for d, v in folds[fold].items():
            p = {tuple(x) for x in pred[d]}
            for s, e, l, *_ in v["gold"]:
                if (s, e, l) not in p:
                    miss.append((d, s, e, l, v["text"][max(0, s - 15):e + 15]))
        rep[fold] = dict(upper_bound_macro_f1=sc["macro_f1"], n_gold=sum(len(v["gold"]) for v in folds[fold].values()),
                         n_unrecoverable=len(miss), examples=miss[:12])
    ntok = {f: int(sum(len(v["ids"]) for v in folds[f].values())) for f in folds}
    rep["tokens_per_fold"] = ntok
    return rep


# ============================================================== torch parts
import torch                                   # noqa: E402
import torch.nn as nn                          # noqa: E402
import torch.nn.functional as F                # noqa: E402


def _import(path, name):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def load_phase1(txl_model_py, ckpt_path):
    """Transformer-XL Phase-1 checkpoint: {'model': state_dict, 'config': dict}."""
    mod = _import(txl_model_py, "txl_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = mod.TXLConfig(**ck["config"])
    m = mod.TransformerXL(cfg)
    m.load_state_dict(ck["model"])
    return m, mod


def load_decoder(legal_lm_py, weights_path):
    """Decoder-only LegalLM Phase-1 weights (run1/final/model.pt + model_config.json)."""
    mod = _import(legal_lm_py, "legal_lm_p2")
    m = mod.load_legal_lm(weights_path, device="cpu", dtype=torch.float32)
    return m.train(), mod


def load_encdec(edmodel_py, ckpt_path):
    """Encoder-decoder Phase-1 checkpoint (phase1_encdec_v2/best.pt: {'model', 'cfg', ...})."""
    mod = _import(edmodel_py, "edmodel_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.EncDecLM(mod.EDConfig(**ck["cfg"]))
    m.load_state_dict(ck["model"])
    return m, mod


def load_retnet(ret_model_py, ckpt_path):
    """RetNet Phase-1 checkpoint (retnet/ckpt/best.pt: {'model', 'config': {'model': ...}, 'vocab_size', ...})."""
    mod = _import(ret_model_py, "ret_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.RetNet(ck["vocab_size"], **ck["config"]["model"])
    sd = ck["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    m.load_state_dict(sd)
    return m, mod


# ---- Archit-2-v2 encoder-decoder (same module/parameter names as the Phase-1 notebook, generation code removed)
class _A2Emb(nn.Module):
    def __init__(self, vocab_size, d_model, max_length, pad_id, dropout):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, ids, offset=0):
        T = ids.size(1)
        pos = torch.arange(offset, offset + T, device=ids.device).unsqueeze(0)
        return self.dropout(self.token_embedding(ids) * math.sqrt(self.d_model) + self.position_embedding(pos))


class _A2MHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        self.d_model, self.h, self.dh, self.p = d_model, num_heads, d_model // num_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.dh).transpose(1, 2)

    def forward(self, query, key_value=None, key_padding_mask=None, causal=False):
        kv = query if key_value is None else key_value
        q, k, v = self._split(self.q_proj(query)), self._split(self.k_proj(kv)), self._split(self.v_proj(kv))
        mask = None
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].bool()
            if causal:
                cm = torch.tril(torch.ones(query.size(1), kv.size(1), dtype=torch.bool, device=query.device))[None, None]
                mask = mask & cm
                causal = False
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.p if self.training else 0.0, is_causal=causal)
        B, _, T, _ = o.shape
        return self.out_proj(o.transpose(1, 2).contiguous().view(B, T, self.d_model))


class _A2FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))


class _A2EncLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.ffn, self.dropout2 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, src_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=src_mask))
        return x + self.dropout2(self.ffn(self.norm2(x)))


class _A2DecLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.cross_attn, self.dropout2 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm3, self.ffn, self.dropout3 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, enc_out, src_mask, self_mask=None):
        """self_mask=None -> causal (= Phase-1). Otherwise a key-padding mask -> bidirectional decoder."""
        if self_mask is None:
            x = x + self.dropout1(self.self_attn(self.norm1(x), causal=True))
        else:
            x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=self_mask))
        x = x + self.dropout2(self.cross_attn(self.norm2(x), key_value=enc_out, key_padding_mask=src_mask))
        return x + self.dropout3(self.ffn(self.norm3(x)))


class _A2Stack(nn.Module):
    def __init__(self, emb, layers, d):
        super().__init__()
        self.embedding, self.layers, self.final_norm = emb, nn.ModuleList(layers), nn.LayerNorm(d)


class A2V2EncDec(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, heads=12, enc_layers=6, dec_layers=8, d_ff=3072,
                 dropout=0.05, enc_max=1024, dec_max=256, pad=0):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, d_model=d_model, heads=heads, enc_layers=enc_layers,
                        dec_layers=dec_layers, d_ff=d_ff, enc_max=enc_max, dec_max=dec_max)
        ee = _A2Emb(vocab_size, d_model, enc_max, pad, dropout)
        de = _A2Emb(vocab_size, d_model, dec_max, pad, dropout)
        de.token_embedding.weight = ee.token_embedding.weight
        self.encoder = _A2Stack(ee, [_A2EncLayer(d_model, heads, d_ff, dropout) for _ in range(enc_layers)], d_model)
        self.decoder = _A2Stack(de, [_A2DecLayer(d_model, heads, d_ff, dropout) for _ in range(dec_layers)], d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = de.token_embedding.weight

    def encode(self, ids, src_mask):
        x = self.encoder.embedding(ids)
        for l in self.encoder.layers:
            x = l(x, src_mask)
        return self.encoder.final_norm(x)

    def decode_hidden(self, ids, enc, src_mask):
        x = self.decoder.embedding(ids)
        for l in self.decoder.layers:
            x = l(x, enc, src_mask)
        return self.decoder.final_norm(x)


def load_encdec2(ckpt_path):
    """Archit-2-v2 best_model.pt (a plain state_dict). Shapes are read from the weights (v2: 768/6/8/3072; v1: 512/6/6/2048)."""
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model" in sd and not any(k.startswith("encoder.") for k in sd):
        sd = sd["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    V, d = sd["encoder.embedding.token_embedding.weight"].shape
    ne = len({k.split(".")[2] for k in sd if k.startswith("encoder.layers.")})
    nd = len({k.split(".")[2] for k in sd if k.startswith("decoder.layers.")})
    m = A2V2EncDec(vocab_size=V, d_model=d, heads=d // 64, enc_layers=ne, dec_layers=nd,
                   d_ff=sd["encoder.layers.0.ffn.fc1.weight"].shape[0],
                   enc_max=sd["encoder.embedding.position_embedding.weight"].shape[0],
                   dec_max=sd["decoder.embedding.position_embedding.weight"].shape[0])
    m.load_state_dict(sd, strict=True)
    return m


def load_backbone(J):
    if J.get("arch", "txl") == "encdec2":
        return load_encdec2(J["phase1_ckpt"])
    if J.get("arch", "txl") == "ret":
        return load_retnet(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "encdec":
        return load_encdec(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "dec":
        return load_decoder(J["model_py"], J["phase1_ckpt"])[0]
    return load_phase1(J.get("model_py", J.get("txl_model_py")), J["phase1_ckpt"])[0]


class LowRankLinear(nn.Module):
    """mode='lora'  : y = W x + b + s * B(A x)
       mode='lorma' : y = (I + s * B A) (W x) + b       (LoRMA_+, multiplicative, identity at init)"""

    def __init__(self, base: nn.Linear, r, alpha, dropout, mode):
        super().__init__()
        self.base, self.mode, self.s = base, mode, alpha / r
        for p in base.parameters():
            p.requires_grad_(False)
        din = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, din))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == "lora":
            return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.s * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class Adapter(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a sub-layer projection."""

    def __init__(self, base: nn.Linear, bottleneck):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


class BiasLinear(nn.Module):
    """BitFit for bias-free projections: frozen W plus a trainable, zero-initialised bias."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return self.base(x) + self.bias


class TaggerHead(nn.Module):
    def _init_head(self, d, n_tags, dropout, head):
        self.drop = nn.Dropout(dropout)
        self.head_type = head
        if head == "bilstm":
            self.lstm = nn.LSTM(d, 256, batch_first=True, bidirectional=True)
            self.head = nn.Linear(512, n_tags)
        else:
            self.head = nn.Linear(d, n_tags)

    def forward(self, ids, attn_mask):
        h = self.hidden(ids, attn_mask)
        if self.head_type == "bilstm":
            h, _ = self.lstm(self.drop(h))
        return self.head(self.drop(h))


class TXLTagger(TaggerHead):
    """Transformer-XL (Phase-1 weights) used as a token encoder.
    attn='bidir' : every token sees left AND right context; relative distances are extended to negative
                   values (the same sinusoid / r_net / v-bias parametrisation, just evaluated at d<0).
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.pos_emb, self.ln_f = base.emb, base.pos_emb, base.ln_f
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.final_norm = self.ln_f
        self._init_head(self.d, n_tags, dropout, head)

    def _block(self, blk, h, r, gidx, mask):
        a = blk.attn
        B, L, _ = h.shape
        H, dh = a.H, a.dh
        x = blk.ln1(h)
        q = a.q_net(x).view(B, L, H, dh).transpose(1, 2)
        kv = a.kv_net(x).view(B, L, 2, H, dh)
        k, v = kv[:, :, 0].transpose(1, 2), kv[:, :, 1].transpose(1, 2)
        rk = a.r_net(r).view(-1, H, dh).transpose(0, 1).unsqueeze(0)          # [1,H,2L-1,dh]
        qu = (q + a.u[None, :, None, :]) * a.scale
        qv = (q + a.v[None, :, None, :]) * a.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                              # [B,H,L,L]
        BDf = torch.matmul(qv, rk.transpose(-1, -2))                            # [B,H,L,2L-1]
        BD = BDf.gather(-1, gidx.expand(B, H, L, L))
        score = (AC + BD).float().masked_fill(mask, -1e30)
        prob = a.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, L, H * dh)
        h = h + a.drop(a.o_net(out))
        h = h + blk.drop(blk.ff2(blk.drop(F.gelu(blk.ff1(blk.ln2(h))))))
        return h

    def hidden(self, ids, attn_mask):
        """ids [B,L] long, attn_mask [B,L] bool (True = real token) -> final hidden [B,L,D]."""
        B, L = ids.shape
        dev = ids.device
        h = self.emb(ids)
        pos = torch.arange(L - 1, -L, -1, device=dev, dtype=torch.float32)       # distance d = L-1-c
        r = self.pos_emb(pos)
        i = torch.arange(L, device=dev)
        gidx = ((L - 1) - i[:, None] + i[None, :])[None, None]                   # c for distance i-j
        mask = (~attn_mask)[:, None, None, :]                                    # [B,1,1,L]
        if self.attn_mode == "causal":
            mask = mask | torch.ones(L, L, dtype=torch.bool, device=dev).triu(1)[None, None]
        for blk in self.blocks:
            h = self._block(blk, h, r, gidx, mask)
        return self.ln_f(h)


class DecTagger(TaggerHead):
    """Decoder-only LegalLM (RoPE, RMSNorm, SwiGLU; Phase-1 weights) used as a token encoder.
    attn='bidir' : causal mask removed (RoPE is relative, so left and right distances are both defined)
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.cfg = base.cfg
        self.tok_emb, self.norm = base.tok_emb, base.norm
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        self.rope_mod = sys.modules[type(base).__module__]
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.norm
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        B, T = ids.shape
        dev = ids.device
        hd = self.cfg.d_model // self.cfg.n_head
        pos = torch.arange(T, device=dev)[None]
        cos, sin = self.rope_mod.rope_cos_sin(pos, hd, self.cfg.rope_theta)
        allow = attn_mask[:, None, None, :]                                   # [B,1,1,T] True = may attend
        if self.attn_mode == "causal":
            allow = allow & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        allow = allow | torch.eye(T, dtype=torch.bool, device=dev)[None, None]  # pad rows never fully masked
        x = self.tok_emb(ids)
        for blk in self.blocks:
            a = blk.attn
            hx = blk.attn_norm(x)
            q = a.q_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            k = a.k_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            v = a.v_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            q, k = self.rope_mod.apply_rope(q, cos, sin), self.rope_mod.apply_rope(k, cos, sin)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=allow)
            x = x + blk.drop(a.o_proj(y.transpose(1, 2).reshape(B, T, -1)))
            x = x + blk.drop(blk.mlp(blk.mlp_norm(x)))
        return self.norm(x)


class EncTagger(TaggerHead):
    """Phase-1 encoder-decoder as a tagger.
    use_decoder=False : encoder only (natively bidirectional).
    use_decoder=True  : BART-style. The window goes through the encoder AND, as decoder input, through the decoder
                        (self-attention + cross-attention to the encoder); tags are read from the DECODER states.
                        attn='bidir' removes the decoder's causal mask, attn='causal' keeps it (= Phase-1 computation).
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        self.cfg = base.c
        self.tok, self.enc_pos, self.enc_norm = base.tok, base.enc_pos, base.enc_norm
        ke = list(range(len(base.enc_layers))) if layers is None else [i for i in layers if i < len(base.enc_layers)]
        self.enc_blocks = nn.ModuleList([base.enc_layers[i] for i in ke])
        if use_decoder:
            self.dec_pos, self.dec_norm = base.dec_pos, base.dec_norm
            kd = list(range(len(base.dec_layers))) if layers is None else [i for i in layers if i < len(base.dec_layers)]
            self.dec_blocks = nn.ModuleList([base.dec_layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))   # for PEFT / partial
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = base.c.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.emb_drop = nn.Dropout(dropout)
        self._init_head(self.d, n_tags, dropout, head)

    def encode(self, ids, attn_mask):
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.enc_pos(torch.arange(T, device=dev))[None])
        mask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal" and not self.use_decoder:
            mask = mask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        mask = mask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.enc_blocks:
            x = l(x, mask)
        return self.enc_norm(x)

    def hidden(self, ids, attn_mask):
        mem = self.encode(ids, attn_mask)
        if not self.use_decoder:
            return mem
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.dec_pos(torch.arange(T, device=dev))[None])
        cmask = attn_mask[:, None, None, :]                                     # cross-attn: real encoder tokens
        smask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal":
            smask = smask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        smask = smask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.dec_blocks:
            x = x + l.drop(l.self_attn(l.n1(x), mask=smask))
            x = x + l.drop(l.cross(l.n2(x), mem=mem, mask=cmask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.dec_norm(x)


class ED2Tagger(TaggerHead):
    """Archit-2-v2 encoder-decoder as a tagger.
    use_decoder=False : tags from the ENCODER (natively bidirectional, 1024 positions). attn must be 'bidir'.
    use_decoder=True  : BART-style. Window -> encoder, same window -> decoder (cross-attends to the encoder);
                        tags from the decoder states. attn='causal' = Phase-1 computation. Decoder has 256 positions.
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        enc = base.encoder
        self.enc_emb, self.enc_norm = enc.embedding, enc.final_norm
        ke = list(range(len(enc.layers))) if layers is None else [i for i in layers if i < len(enc.layers)]
        self.enc_blocks = nn.ModuleList([enc.layers[i] for i in ke])
        if use_decoder:
            dec = base.decoder
            self.dec_emb, self.dec_norm = dec.embedding, dec.final_norm
            kd = list(range(len(dec.layers))) if layers is None else [i for i in layers if i < len(dec.layers)]
            self.dec_blocks = nn.ModuleList([dec.layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = enc.embedding.d_model
        for m in list(self.blocks.modules()) + [self.enc_emb.dropout] + ([self.dec_emb.dropout] if use_decoder else []):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        x = self.enc_emb(ids)
        for l in self.enc_blocks:
            x = l(x, attn_mask)
        mem = self.enc_norm(x)
        if not self.use_decoder:
            return mem
        y = self.dec_emb(ids)
        sm = None if self.attn_mode == "causal" else attn_mask
        for l in self.dec_blocks:
            y = l(y, mem, attn_mask, self_mask=sm)
        return self.dec_norm(y)


class RetTagger(TaggerHead):
    """RetNet (Phase-1 weights) as a token encoder, parallel form of multi-scale retention over a window.
    attn='causal': D_nm = gamma^(n-m) for n>=m  (exactly the Phase-1 computation; verified in the notebook)
    attn='bidir' : D_nm = gamma^|n-m|           (symmetric decay, so every token also retains its right context)
    Padding uses document id -1, so the same-document mask of RetNet also removes pads."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.final_ln = base.emb, base.final_ln
        self.register_buffer("inv_freq", base.inv_freq.clone(), persistent=False)
        keep = list(range(len(base.layers))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.layers[i] for i in keep])
        self.d = base.d_model
        self.rot = sys.modules[type(base).__module__].rotate
        self.emb_drop = nn.Dropout(dropout)
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.final_ln
        self._init_head(self.d, n_tags, dropout, head)

    def _msr(self, m, x, cos, sin, doc):
        B, L, _ = x.shape
        H, dk, dv = m.H, m.dk, m.dv
        q = m.wq(x).view(B, L, H, dk)
        k = m.wk(x).view(B, L, H, dk)
        v = m.wv(x).view(B, L, H, dv)
        g = m.wg(x)
        with torch.autocast(device_type=x.device.type, enabled=False):
            q = (self.rot(q.float(), cos, sin) * dk ** -0.5).transpose(1, 2)
            k = self.rot(k.float(), cos, sin).transpose(1, 2)
            v = v.float().transpose(1, 2)
            i = torch.arange(L, device=x.device, dtype=torch.float32)
            diff = i.view(L, 1) - i.view(1, L)
            lg = m.log_gamma.view(H, 1, 1)
            if self.attn_mode == "causal":
                D = torch.exp(lg * diff.clamp(min=0)) * (diff >= 0)
            else:
                D = torch.exp(lg * diff.abs())
            same = (doc.unsqueeze(2) == doc.unsqueeze(1)).unsqueeze(1)
            o = torch.matmul(torch.matmul(q, k.transpose(-1, -2)) * D.unsqueeze(0) * same, v)
            o = o.transpose(1, 2).reshape(B * L, H * dv)
            o = F.group_norm(o, H, m.gn_w.float(), m.gn_b.float(), 1e-5)
        o = o.view(B, L, H * dv)
        return m.wo(F.silu(g) * o.to(g.dtype))

    def hidden(self, ids, attn_mask):
        B, L = ids.shape
        doc = torch.where(attn_mask, torch.zeros_like(ids), torch.full_like(ids, -1))
        pos = torch.arange(L, device=ids.device)[None].expand(B, L)
        ang = pos.float().unsqueeze(-1) * self.inv_freq
        cos, sin = ang.cos().unsqueeze(2), ang.sin().unsqueeze(2)
        h = self.emb_drop(self.emb(ids))
        for blk in self.blocks:
            h = h + blk.drop(self._msr(blk.msr, blk.ln1(h), cos, sin, doc))
            y = blk.ln2(h)
            h = h + blk.drop(blk.down(F.silu(blk.gate(y)) * blk.up(y)))
        return self.final_ln(h)


def _linear_slots(blk, names):
    """(holder_module, attribute) for every nn.Linear attribute in `names` inside one block."""
    out = []
    for mod in blk.modules():
        for nm in names:
            if isinstance(getattr(mod, nm, None), nn.Linear) and not isinstance(mod, (LowRankLinear, Adapter, BiasLinear)):
                out.append((mod, nm))
    return out


def apply_method(model, method, J):
    """Freeze / wrap according to the adaptation method. The head is always trainable."""
    for p in model.parameters():
        p.requires_grad_(method in ("full", "kd"))
    if method == "partial":
        for blk in list(model.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in model.final_norm.parameters():
            p.requires_grad_(True)
    elif method in ("lora", "lorma"):
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                setattr(holder, nm, LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method))
    elif method == "adapter":
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J.get("adapter_targets", ["o_net", "ff2"])):
                setattr(holder, nm, Adapter(getattr(holder, nm), J["adapter_dim"]))
    elif method == "bitfit":
        if J.get("arch", "txl") in ("dec", "ret"):            # bias-free model: add zero-init biases + norms
            for blk in model.blocks:
                for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                    setattr(holder, nm, BiasLinear(getattr(holder, nm)))
        for n, p in model.named_parameters():
            if n.endswith("bias") or n.endswith(".u") or n.endswith(".v") or n.endswith(".gn_w") or n.endswith(".gn_b"):
                p.requires_grad_(True)
        for m in model.modules():                              # every LayerNorm / RMSNorm
            if "Norm" in type(m).__name__:
                for p in m.parameters():
                    p.requires_grad_(True)
    for p in model.head.parameters():
        p.requires_grad_(True)
    if getattr(model, "lstm", None) is not None:
        for p in model.lstm.parameters():
            p.requires_grad_(True)
    return model


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# ============================================================== windows / batching
def windows_for(n, L, stride):
    if n <= L:
        return [0]
    st = list(range(0, n - L + 1, stride))
    if st[-1] + L < n:
        st.append(n - L)
    return st


def make_train_windows(docs, L, stride):
    W = []
    for d in docs.values():
        n = len(d["ids"])
        for s in windows_for(n, L, stride):
            W.append((d["ids"][s:s + L], d["y"][s:s + L]))
    return W


def collate(items, with_y=True):
    L = max(len(x[0]) for x in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    am = torch.zeros(len(items), L, dtype=torch.bool)
    y = torch.full((len(items), L), -100, dtype=torch.long)
    for b, it in enumerate(items):
        n = len(it[0])
        ids[b, :n] = torch.as_tensor(it[0], dtype=torch.long)
        am[b, :n] = True
        if with_y:
            y[b, :n] = torch.as_tensor(it[1], dtype=torch.long)
    return ids, am, y


@torch.no_grad()
def doc_logprobs(model, docs, L, stride, bs, device, amp=True):
    """Merge per-window log-probs into one [n, T] array per doc (triangular weights: tokens near a window
    edge get little weight, so every token is mostly judged with context on both sides)."""
    model.eval()
    jobs = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n:
            for s in windows_for(n, L, stride):
                jobs.append((did, s, min(L, n - s)))
    T = model.head.out_features
    acc = {did: np.zeros((len(d["ids"]), T), np.float32) for did, d in docs.items()}
    wsum = {did: np.zeros((len(d["ids"]), 1), np.float32) for did, d in docs.items()}
    jobs.sort(key=lambda j: -j[2])
    for k in range(0, len(jobs), bs):
        chunk = jobs[k:k + bs]
        items = [(docs[did]["ids"][s:s + n], None) for did, s, n in chunk]
        ids, am, _ = collate(items, with_y=False)
        ids, am = ids.to(device), am.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            lg = model(ids, am)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for b, (did, s, n) in enumerate(chunk):
            pos = np.arange(n)
            w = np.minimum(pos + 1, n - pos).astype(np.float32)[:, None]
            if s == 0:
                w[: n // 2] = np.maximum(w[: n // 2], n / 2)        # doc start: no left context exists anyway
            if s + n == len(docs[did]["ids"]):
                w[n // 2:] = np.maximum(w[n // 2:], n / 2)          # doc end
            acc[did][s:s + n] += lp[b, :n] * w
            wsum[did][s:s + n] += w
    return {did: acc[did] / np.maximum(wsum[did], 1e-6) for did in docs}


def decode_docs(docs, lps, tags, use_viterbi=True):
    allowed, start_ok = bio_transitions(tags)
    pred = {}
    for did, d in docs.items():
        lp = lps[did]
        tg = viterbi(lp, allowed, start_ok) if use_viterbi else lp.argmax(-1)
        pred[did] = spans_from_tags(d["text"], d, tg, tags)
    return pred


# ============================================================== training
def class_weights(windows, T, mode):
    if mode == "none":
        return None
    cnt = np.bincount(np.concatenate([w[1] for w in windows]), minlength=T).astype(np.float64)
    w = np.ones(T)
    ent = cnt[1:]
    mean = ent[ent > 0].mean()
    w[1:] = np.clip(np.sqrt(mean / np.maximum(ent, 1)), 0.5, 4.0)
    return torch.tensor(w, dtype=torch.float32)


def build_model(J, base_factory, n_tags, method):
    base = base_factory()
    layers = J["kd_student_layers"] if method == "kd" else None
    cls = {"dec": DecTagger, "encdec": EncTagger, "encdec2": ED2Tagger, "ret": RetTagger}.get(J.get("arch", "txl"), TXLTagger)
    kw = dict(use_decoder=J.get("encdec_use_decoder", False)) if cls in (EncTagger, ED2Tagger) else {}
    if cls is ED2Tagger and not kw["use_decoder"]:
        assert J["attn"] == "bidir", "encoder-only tagging is always bidirectional"
    m = cls(base, n_tags, attn=J["attn"], layers=layers, dropout=J["dropout"], head=J["head"], **kw)
    return apply_method(m, method, J)


def train_one(J, model, train_docs, val_docs, tags, labels, device, epochs, log, teacher=None):
    torch.manual_seed(J["seed"]); np.random.seed(J["seed"])
    model.to(device)
    W = make_train_windows(train_docs, J["max_len"], J["train_stride"])
    cw = class_weights(W, len(tags), J["class_weight"])
    cw = cw.to(device) if cw is not None else None
    head_names = {n for n, _ in model.named_parameters() if n.startswith("head") or n.startswith("lstm")}
    new_ids = set()
    for mm in model.modules():
        if isinstance(mm, LowRankLinear):
            new_ids |= {id(mm.A), id(mm.B)}
        elif isinstance(mm, Adapter):
            new_ids |= {id(p) for p in list(mm.down.parameters()) + list(mm.up.parameters())}
    new_names = {n for n, p in model.named_parameters() if id(p) in new_ids}
    g_head = [p for n, p in model.named_parameters() if p.requires_grad and (n in head_names or n in new_names)]
    g_body = [p for n, p in model.named_parameters() if p.requires_grad and n not in head_names and n not in new_names]
    groups = [dict(params=g_head, lr=J["head_lr"])]
    if g_body:
        groups.append(dict(params=g_body, lr=J["lr"], is_body=True))
    opt = torch.optim.AdamW(groups, weight_decay=J["weight_decay"])
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    steps_per_epoch = math.ceil(len(W) / J["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(0.06 * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)   # low start: fewer skipped fp16 steps
    n_skip = 0
    best = dict(f1=-1, epoch=0, state=None, report=None, pred=None)
    hist, step, bad = [], 0, 0
    t0 = time.time()
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    for ep in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(len(W))
        tl, tn, gsum = 0.0, 0, [0.0]
        for k in range(0, len(W), J["batch_size"]):
            ids, am, y = collate([W[i] for i in perm[k:k + J["batch_size"]]])
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            f = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))   # warmup + linear decay
            # LP-FT: optionally keep the pretrained body frozen for the first epoch(s) so the random head
            # does not distort the Phase-1 features, then re-warm the body learning rate.
            nf = J.get("freeze_body_epochs", 0)
            if ep <= nf:
                fb = 0.0
            else:
                bs = step - nf * steps_per_epoch
                fb = min(1.0, (bs + 1) / warm) * (1.0 if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * (fb if (g.get("is_body") and nf) else f)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lg = model(ids, am)
            lg = lg.float()
            loss = F.cross_entropy(lg.view(-1, lg.size(-1)), y.view(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    tl_ = teacher(ids, am).float()
                Tt = J["kd_temperature"]
                kl = F.kl_div(F.log_softmax(lg / Tt, -1), F.log_softmax(tl_ / Tt, -1), log_target=True, reduction="none").sum(-1)
                kl = (kl * am).sum() / am.sum()
                loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * (Tt ** 2) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            # clip each parameter group separately: a large body gradient must never shrink the head's update
            gn = [float(torch.nn.utils.clip_grad_norm_(g["params"], 1.0)) for g in opt.param_groups]
            gsum = [a + b for a, b in zip(gsum, gn)] if tn else gn
            sc0 = scaler.get_scale() if use_amp else 1.0
            scaler.step(opt); scaler.update()
            if use_amp and scaler.get_scale() < sc0:
                n_skip += 1                                     # overflow -> this optimizer step was skipped
            tl += loss.item(); tn += 1; step += 1
        rec = dict(epoch=ep, train_loss=tl / max(tn, 1), secs=time.time() - t0, skipped_steps=n_skip,
                   grad_norm_head=round(gsum[0] / max(tn, 1), 3),
                   grad_norm_body=round(gsum[1] / max(tn, 1), 3) if len(gsum) > 1 else None)
        if val_docs:
            lps = doc_logprobs(model, val_docs, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
            pred = decode_docs(val_docs, lps, tags)
            sc = strict_scores({d: v["gold"] for d, v in val_docs.items()}, pred, labels)
            rec.update(val_f1=sc["macro_f1"], val_p=sc["macro_p"], val_r=sc["macro_r"])
            if sc["macro_f1"] > best["f1"]:
                best.update(f1=sc["macro_f1"], epoch=ep, report=sc, pred=pred,
                            state={n: p.detach().cpu().clone() for n, p in model.state_dict().items()})
                bad = 0
            else:
                bad += 1
        hist.append(rec)
        log(json.dumps(rec))
        if val_docs and bad >= J["patience"]:
            log(f"early stop at epoch {ep} (best epoch {best['epoch']}, val F1 {best['f1']:.4f})")
            break
    info = dict(train_secs=time.time() - t0, history=hist, n_windows=len(W), steps=step,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2**30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    labels, tags, folds = D["labels"], D["tags"], D["folds"]
    if J.get("quick"):
        folds = {f: dict(list(v.items())[: J["quick_docs"]]) for f, v in folds.items()}
    method = J["method"]
    base_factory = lambda: load_backbone(J)
    log(f"arch={J.get('arch', 'txl')} method={method} attn={J['attn']} head={J['head']} device={device}")

    def get_teacher(which):
        tp = Path(J["kd_teacher_dir"]) / f"{which}_model.pt"
        assert tp.exists(), f"KD needs the teacher checkpoint {tp} (run the teacher job first)"
        tj = json.load(open(Path(J["kd_teacher_dir"]) / "job.json"))
        t = build_model(tj, base_factory, len(tags), tj["method"])
        t.load_state_dict(torch.load(tp, map_location="cpu"))
        t.to(device).eval()
        for p in t.parameters():
            p.requires_grad_(False)
        return t

    results = dict(name=J["name"], method=method, attn=J["attn"], head=J["head"], config=J)
    # ---- (1) model selection: train fold1 -> validate fold2
    model = build_model(J, base_factory, len(tags), method)
    results["params_total"], results["params_trainable"] = count_params(model)
    log(f"params total={results['params_total']/1e6:.2f}M trainable={results['params_trainable']/1e6:.3f}M")
    teacher = get_teacher("val") if method == "kd" else None
    if teacher is not None:
        results["teacher_params"] = count_params(teacher)[0]
    best, info = train_one(J, model, folds["fold1"], folds["fold2"], tags, labels, device, J["epochs"], log, teacher)
    results.update(val_macro_f1=best["f1"], best_epoch=best["epoch"], val_report=best["report"], val_train=info,
                   val_errors=error_breakdown({d: v["gold"] for d, v in folds["fold2"].items()}, best["pred"]))
    json.dump(best["pred"], open(out / "val_pred_fold2.json", "w"))
    log(f"VALIDATION fold1->fold2 strict macro-F1 = {best['f1']:.4f} (epoch {best['epoch']})")
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    del teacher
    # ---- (2) final model
    if J["final_fit"] == "all":
        epochs = max(1, best["epoch"] + J.get("final_extra_epochs", 0))
        model = build_model(J, base_factory, len(tags), method)
        teacher = get_teacher("final") if method == "kd" else None
        train_docs = {**folds["fold1"], **folds["fold2"]}
        _, finfo = train_one(J, model, train_docs, None, tags, labels, device, epochs, log, teacher)
        results["final_train"] = finfo
        del teacher
    else:
        model.load_state_dict(best["state"])
    model.to(device)
    if J["save_model"]:
        torch.save({n: p.detach().cpu() for n, p in model.state_dict().items()}, out / "final_model.pt")
    # ---- (3) test prediction
    t0 = time.time()
    test = folds["fold3"]
    lps = doc_logprobs(model, test, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
    pred = decode_docs(test, lps, tags)
    results["test_infer_secs"] = time.time() - t0
    all_ids = list(D["folds"]["fold3"].keys())
    final = {did: [[int(s), int(e), l] for s, e, l in pred.get(did, [])] for did in all_ids}
    assert set(final) == set(all_ids)
    for did, spans in final.items():
        n = len(D["folds"]["fold3"][did]["text"])
        assert all(0 <= s < e <= n and l in labels for s, e, l in spans), did
    pj = out / "prediction.json"
    json.dump(final, open(pj, "w"))
    zp = out / f"submission_{J['name']}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    results["test_entities"] = sum(len(v) for v in final.values())
    results["test_label_counts"] = dict(Counter(l for v in final.values() for _, _, l in v))
    json.dump(results, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> {zp} | test entities={results['test_entities']} | {json.dumps(results['test_label_counts'])}")


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p2/p2_ner.py


In [3]:
# ============================================================ 2. PREP + ALIGNMENT CHECK + MODEL CHECK
sys.path.insert(0, str(WORK))
import importlib, p2_ner as P; importlib.reload(P)
import torch, numpy as np

tok = P.load_bpe(TOK_PY, TOK_PATH, norm=NORM)
print("tokenizer kind:", tok.kind, "| normalisation:", NORM, "| vocab", tok.vocab_size); assert tok.vocab_size == 32000
CACHE = WORK / "cache.pkl"
t0 = time.time()
folds = P.prepare(DATA, tok, LABELS, CACHE)
print(f"tokenised all folds in {time.time()-t0:.0f}s | docs needing the slow exact-alignment fallback: {getattr(tok, 'n_fallback', 0)}")

rep = P.alignment_report(folds, LABELS)
for f in ["fold1", "fold2"]:
    r = rep[f]
    print(f"{f}: alignment upper-bound strict macro-F1 = {r['upper_bound_macro_f1']:.4f} | gold {r['n_gold']} | unrecoverable {r['n_unrecoverable']}")
    for ex in r["examples"][:6]: print("    ", ex)
print("tokens per fold:", rep["tokens_per_fold"])
assert rep["fold1"]["upper_bound_macro_f1"] > 0.97, "alignment is losing too many spans - send me this output"
# token ids must be exactly what Phase-1 pretraining produced for this text
for fold in ["fold1", "fold3"]:
    for d in list(folds[fold].values())[:8]:
        assert list(d["ids"]) == list(tok.reference_encode(d["text"])), "token ids differ from the Phase-1 tokenizer"
print("token ids identical to Phase-1 tokenisation: OK")

# encoder in causal mode == Phase-1 forward (on a real document window)
J0 = dict(arch=ARCH, model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT)
base = P.load_backbone(J0).eval()
n_par = sum(p.numel() for p in base.parameters())
print(f"Phase-1 model: {n_par/1e6:.2f}M params | config {getattr(base, 'cfg', getattr(base, 'c', None))}")
d0 = next(iter(folds["fold1"].values()))
ids = torch.as_tensor(d0["ids"][:256], dtype=torch.long)[None]
am = torch.ones_like(ids, dtype=torch.bool)
with torch.no_grad():
    if ARCH == "txl":
        tg = P.TXLTagger(base, 25, attn="causal", dropout=0.0).eval()
        ref = base(ids)["logits"]
        mine = torch.nn.functional.linear(tg.hidden(ids, am), base.emb.weight, base.out_bias)
    elif ARCH == "dec":
        tg = P.DecTagger(base, 25, attn="causal", dropout=0.0).eval()
        ref = base.hidden_states(ids); mine = tg.hidden(ids, am)
    elif ARCH == "ret":
        tg = P.RetTagger(base, 25, attn="causal", dropout=0.0).eval()
        S0, pd0 = base.init_state(1, ids.device)
        ref = base.hidden(ids, torch.zeros_like(ids), torch.arange(ids.size(1))[None], S0, pd0)[0]; mine = tg.hidden(ids, am)
    elif ARCH == "encdec2":                            # encoder tagging == Phase-1 encoder; decoder tagging (causal) == Phase-1 decoder
        mem0 = base.encode(ids, am)
        tg = P.ED2Tagger(base, 25, attn="bidir", dropout=0.0).eval(); mine = tg.hidden(ids, am); ref = mem0
        tgd = P.ED2Tagger(base, 25, attn="causal", dropout=0.0, use_decoder=True).eval()
        d2 = (base.decode_hidden(ids, mem0, am) - tgd.hidden(ids, am)).abs().max().item()
        print("decoder-tagging path vs Phase-1 decoder, max |diff| =", d2); assert d2 < 1e-3
    else:                                              # encoder-decoder, BART-style decoder tagging (causal = Phase-1 computation)
        tg = P.EncTagger(base, 25, attn="causal", dropout=0.0, use_decoder=True).eval()
        mem0, msk0 = base.encode(ids); ref = base.decode_hidden(ids, mem0, msk0); mine = tg.hidden(ids, am)
diff = (ref - mine).abs().max().item(); print("tagger encoder vs Phase-1 forward, max |diff| =", diff); assert diff < 1e-3
del base, tg


tokenizer kind: hf | normalisation: ws | vocab 32000
tokenised all folds in 3s | docs needing the slow exact-alignment fallback: 0
fold1: alignment upper-bound strict macro-F1 = 0.9890 | gold 4120 | unrecoverable 67
     ('975074', 7995, 8085, 'PREC', '6 SC 1565] and Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]. Even otherwis')
     ('736324', 1963, 1976, 'AUTH', 'truck down the Railway Boards instructions ')
     ('736324', 11218, 11231, 'AUTH', 'lementation of Railway Boards decision to e')
     ('736324', 12849, 12901, 'PREC', 'rlier cases in Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]. Para 7 of the')
     ('736324', 14121, 14174, 'PREC', ' 4 SCC 268] or Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]. Hence the con')
     ('1243363', 17853, 17865, 'STAT', 'ns". After the Constitutional Amendment (4')
fold2: alignment upper-bound strict macro-F1 = 0.9848 | gold 4273 | unrecoverable 85
     ('76640285', 18, 41, 'COURT', 'O

In [4]:
# ============================================================ 3. EXPERIMENTS
COMMON = dict(
    arch=ARCH, cache=str(CACHE), model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT,
    attn="bidir",              # 'bidir' (sees both sides; recommended) | 'causal' (ablation)
    head="linear",             # 'linear' | 'bilstm'
    max_len=512, train_stride=256, eval_stride=256, batch_size=8, eval_batch_size=16,
    encdec_use_decoder=False,   # tag from the bidirectional ENCODER (job "full_dec" tries decoder tagging instead)
    lowrank_targets=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"], adapter_targets=["out_proj", "fc2"],
    kd_student_layers=[0, 2, 5],                # 3 of 6 encoder layers
    lr=1e-4,
    head_lr=1e-3, weight_decay=0.01, dropout=0.1,
    epochs=15, patience=5, class_weight="sqrt", seed=781,
    rank=16, alpha=32, lowrank_dropout=0.05, adapter_dim=64, partial_top_layers=4,
    kd_temperature=2.0, kd_alpha=0.5,
    final_fit="all",           # 'all': retrain on fold1+fold2 for best-epoch epochs | 'best': use the fold1 model
    save_model=True, quick=QUICK_TEST, quick_docs=4,
)
#            name        method     overrides
JOBS = {
  "full":    ("full",    dict()),                                              # also the KD teacher
  "lorma":   ("lorma",   dict(epochs=25, save_model=False)),                   # compulsory
  "lora":    ("lora",    dict(epochs=25, save_model=False)),
  "kd":      ("kd",      dict(lr=2e-4, kd_teacher_dir=str(OUT / "full"), save_model=False)),
  "adapter": ("adapter", dict(epochs=25, save_model=False)),
  "bitfit":  ("bitfit",  dict(epochs=25, lr=2e-3, save_model=False)),
  "full_dec": ("full", dict(encdec_use_decoder=True, attn="causal", max_len=256, train_stride=128, eval_stride=128, save_model=False)),   # ablation: BART-style decoder tagging
  # extra ablations (uncomment if time allows):
  # "lorma_causal": ("lorma", dict(attn="causal", epochs=25, save_model=False)),
  # "full_bilstm":  ("full",  dict(head="bilstm", save_model=False)),
  # "partial":      ("partial", dict(lr=2e-4, save_model=False)),
}
# two GPUs -> two queues (kd must run after full on the same queue)
QUEUES = [["full", "kd", "bitfit"], ["lorma", "lora", "adapter"]]
if "full_causal" in JOBS: QUEUES[1].append("full_causal")
if "full_dec" in JOBS: QUEUES[1].append("full_dec")
if QUICK_TEST:
    QUEUES = [["full", "kd"], ["lorma"] + (["full_dec"] if "full_dec" in JOBS else [])]

n_gpu = torch.cuda.device_count(); print("GPUs:", n_gpu)
if n_gpu < 2: QUEUES = [sum(QUEUES, [])]

def write_job(name):
    meth, over = JOBS[name]
    J = dict(COMMON, name=name, method=meth, out_dir=str(OUT / name), **over)
    if QUICK_TEST: J.update(epochs=2, patience=2)
    Path(J["out_dir"]).mkdir(parents=True, exist_ok=True)
    p = Path(J["out_dir"]) / "job.json"; json.dump(J, open(p, "w"), indent=1)
    return p

def run_queue(names, gpu):
    for n in names:
        if (OUT / n / "results.json").exists():
            print(f"[skip] {n} already done"); continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
        p = subprocess.Popen([sys.executable, str(WORK / "p2_ner.py"), str(write_job(n))], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            if "Warning" not in line: print(f"[gpu{gpu}] {line}", end="")
        p.wait()
        if p.returncode != 0:
            print(f"[gpu{gpu}] !!! job {n} FAILED (see {OUT/n/'log.txt'} and the lines above)")

T0 = time.time()
threads = [threading.Thread(target=run_queue, args=(q, g)) for g, q in enumerate(QUEUES)]
for t in threads: t.start()
for t in threads: t.join()
print(f"all jobs finished in {(time.time()-T0)/60:.1f} min")


GPUs: 2
[gpu0] [11:38:05][full] arch=encdec2 method=full attn=bidir head=linear device=cuda
[gpu1] [11:38:05][lorma] arch=encdec2 method=lorma attn=bidir head=linear device=cuda
[gpu0] [11:38:07][full] params total=67.91M trainable=67.910M
[gpu1] [11:38:08][lorma] params total=69.24M trainable=1.346M
[gpu0] [11:38:34][full] {"epoch": 1, "train_loss": 1.0171182345260272, "secs": 13.714858770370483, "skipped_steps": 0, "grad_norm_head": 1.092, "grad_norm_body": 1.09, "val_f1": 0.08050363854477428, "val_p": 0.08410106971905895, "val_r": 0.08085865670387293}
[gpu1] [11:38:36][lorma] {"epoch": 1, "train_loss": 1.0879242670806972, "secs": 14.468904256820679, "skipped_steps": 0, "grad_norm_head": 1.397, "grad_norm_body": null, "val_f1": 0.0, "val_p": 0.0, "val_r": 0.0}
[gpu0] [11:38:53][full] {"epoch": 2, "train_loss": 0.20225723311305047, "secs": 32.9794442653656, "skipped_steps": 0, "grad_norm_head": 0.244, "grad_norm_body": 0.295, "val_f1": 0.11279800087763377, "val_p": 0.1193530637120344,

In [5]:
# ============================================================ 4. SUMMARY + SUBMISSIONS
import pandas as pd, zipfile
rows, per = [], {}
SUB = Path("/kaggle/working/submissions" + ("_quick" if QUICK_TEST else "")); SUB.mkdir(exist_ok=True)
test_ids = set(json.load(open(DATA / "Test" / "fold3-text.json")).keys())
for n in JOBS:
    rp = OUT / n / "results.json"
    if not rp.exists(): continue
    r = json.load(open(rp))
    z = OUT / n / f"submission_{n}.zip"
    with zipfile.ZipFile(z) as zz:
        assert zz.namelist() == ["prediction.json"]
        pj = json.loads(zz.read("prediction.json")); assert set(pj) == test_ids
    shutil.copy(z, SUB / z.name)
    rows.append(dict(exp=n, method=r["method"], attn=r["attn"], head=r["head"],
                     val_macroF1=round(r["val_macro_f1"], 4), val_P=round(r["val_report"]["macro_p"], 4), val_R=round(r["val_report"]["macro_r"], 4),
                     best_epoch=r["best_epoch"], params_M=round(r["params_total"]/1e6, 2), trainable_M=round(r["params_trainable"]/1e6, 3),
                     peak_mem_GB=round(r["val_train"]["peak_mem_gb"] or 0, 2), train_min=round(r["val_train"]["train_secs"]/60, 1),
                     final_train_min=round(r.get("final_train", {}).get("train_secs", 0)/60, 1), test_infer_s=round(r["test_infer_secs"], 1),
                     test_entities=r["test_entities"], fp_types=r["val_errors"]["fp_kinds"]))
    per[n] = {l: round(v["f1"], 3) for l, v in r["val_report"]["per_label"].items()}
df = pd.DataFrame(rows).sort_values("val_macroF1", ascending=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(df)
display(pd.DataFrame(per).T)            # per-label F1 on fold2, per experiment
df.to_csv(WORK / ("summary_quick.csv" if QUICK_TEST else "summary.csv"), index=False)
print("submission zips:", sorted(p.name for p in SUB.glob("*.zip")), "->", SUB)
if QUICK_TEST: print("\nQUICK TEST ONLY - do NOT submit these. Set QUICK_TEST=False and Save & Run All (Commit).")


,exp,method,attn,head,val_macroF1,val_P,val_R,best_epoch,params_M,trainable_M,peak_mem_GB,train_min,final_train_min,test_infer_s,test_entities,fp_types
6,full_dec,full,causal,linear,0.4241,0.4499,0.4340,13,143.72,143.723,3.72,14.9,20.0,14.0,4055,"{'boundary_error': 703, 'overlap_other_label':..."
3,kd,kd,bidir,linear,0.1526,0.1454,0.1713,6,46.65,46.647,1.54,2.9,2.3,4.8,4827,"{'spurious': 1515, 'overlap_other_label': 815,..."
2,lora,lora,bidir,linear,0.1489,0.1441,0.1665,18,69.24,1.346,1.42,8.4,8.2,8.3,4331,"{'spurious': 1569, 'overlap_other_label': 730,..."
0,full,full,bidir,linear,0.1468,0.1388,0.1674,11,67.91,67.910,2.00,5.0,5.3,7.0,4657,"{'spurious': 1602, 'overlap_other_label': 811,..."
4,adapter,adapter,bidir,linear,0.1359,0.1387,0.1430,10,69.10,1.209,1.00,4.2,3.3,7.2,4167,"{'spurious': 1257, 'label_error': 64, 'overlap..."
5,bitfit,bitfit,bidir,linear,0.0984,0.1124,0.0936,4,67.91,0.081,0.95,2.4,1.3,6.9,2940,"{'overlap_other_label': 440, 'spurious': 807, ..."
1,lorma,lorma,bidir,linear,0.0960,0.1033,0.0938,8,69.24,1.346,1.60,5.1,4.0,9.3,0,"{'spurious': 708, 'boundary_error': 844, 'over..."


,APP,RESP,A.COUNSEL,R.COUNSEL,JUDGE,WIT,AUTH,COURT,STAT,PREC,DATE,CASENO
full,0.006,0.028,0.072,0.030,0.113,0.000,0.145,0.636,0.317,0.055,0.345,0.014
lorma,0.000,0.000,0.000,0.000,0.000,0.000,0.219,0.527,0.289,0.000,0.117,0.000
lora,0.000,0.012,0.095,0.043,0.079,0.000,0.180,0.612,0.338,0.046,0.359,0.024
kd,0.000,0.007,0.075,0.042,0.075,0.000,0.157,0.687,0.325,0.062,0.378,0.023
adapter,0.024,0.008,0.051,0.046,0.051,0.000,0.192,0.650,0.285,0.049,0.253,0.021
bitfit,0.000,0.000,0.000,0.000,0.011,0.000,0.129,0.650,0.252,0.011,0.128,0.000
full_dec,0.127,0.174,0.483,0.429,0.487,0.044,0.287,0.815,0.639,0.315,0.909,0.381


submission zips: ['submission_adapter.zip', 'submission_bitfit.zip', 'submission_full.zip', 'submission_full_dec.zip', 'submission_kd.zip', 'submission_lora.zip', 'submission_lorma.zip'] -> /kaggle/working/submissions
